# Dataset Preparation

In [ ]:
from datasets import load_dataset, Dataset

raw = load_dataset("DeepMount00/CulturaViva-ITA", split="train")
split_ds = raw.train_test_split(test_size=0.1, seed=42)

train_ds = split_ds["train"]
test_ds = split_ds["test"]


def build_ir_split(ds, split_name):
    queries = []
    corpus = []
    qrels = []

    for i, row in enumerate(ds):
        qid = f"{split_name}_q_{i}"
        did = f"{split_name}_d_{i}"

        queries.append(
            {
                "_id": qid,
                "text": row["prompt"],
            }
        )

        corpus.append(
            {
                "_id": did,
                "title": "",
                "text": row["answer"],
            }
        )

        qrels.append(
            {
                "query-id": qid,
                "corpus-id": did,
                "score": 1,
            }
        )

    return (
        Dataset.from_list(queries),
        Dataset.from_list(corpus),
        Dataset.from_list(qrels),
    )


queries_train, corpus_train, qrels_train = build_ir_split(train_ds, "train")
queries_test, corpus_test, qrels_test = build_ir_split(test_ds, "test")

repo_id = "lopozz/CulturaViva-Retrieval"

# push to repo
# DatasetDict({
#     "train": queries_train,
#     "test": queries_test,
# }).push_to_hub(repo_id, config_name="queries")

# # push corpus as one config
# DatasetDict({
#     "train": corpus_train,
#     "test": corpus_test,
# }).push_to_hub(repo_id, config_name="corpus")

# # push qrels as one config
# DatasetDict({
#     "train": qrels_train,
#     "test": qrels_test,
# }).push_to_hub(repo_id, config_name="qrels")

# Train

In [ ]:
from datasets import load_dataset

repo_id = "lopozz/CulturaViva-Retrieval"


def build_pair_dataset(
    repo_id: str, split: str, max_examples: int | None = None
) -> Dataset:
    queries_ds = load_dataset(repo_id, "queries", split=split)
    corpus_ds = load_dataset(repo_id, "corpus", split=split)
    qrels_ds = load_dataset(repo_id, "qrels", split=split)

    query_text_by_id = {row["_id"]: row["text"] for row in queries_ds}
    doc_text_by_id = {row["_id"]: row["text"] for row in corpus_ds}

    pairs = []
    for row in qrels_ds:
        if int(row["score"]) > 0:
            qid = row["query-id"]
            did = row["corpus-id"]

            if qid in query_text_by_id and did in doc_text_by_id:
                pairs.append(
                    {
                        "query": query_text_by_id[qid],
                        "answer": doc_text_by_id[did],
                    }
                )

                if max_examples is not None and len(pairs) >= max_examples:
                    break

    return Dataset.from_list(pairs)


train_dataset = build_pair_dataset(repo_id, "train", max_examples=100)
eval_dataset = build_pair_dataset(repo_id, "test", max_examples=10)

print(train_dataset)
print(eval_dataset)

In [ ]:
import logging

from sentence_transformers import (
    SparseEncoder,
    SparseEncoderModelCardData,
    SparseEncoderTrainer,
    SparseEncoderTrainingArguments,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.losses import (
    SparseMultipleNegativesRankingLoss,
    SpladeLoss,
)
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

logging.basicConfig(
    format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO
)

# 1. Load a model to finetune with 2. (Optional) model card data
mlm_transformer = Transformer("jhu-clsp/mmBERT-small", transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)


# 4. Define a loss function
loss = SpladeLoss(
    model=model,
    loss=SparseMultipleNegativesRankingLoss(model=model),
    query_regularizer_weight=0,
    document_regularizer_weight=3e-3,
)

batch_size = 1
# 5. (Optional) Specify training arguments
run_name = "inference-free-splade-mmBERT-small-ita"
args = SparseEncoderTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    learning_rate=2e-5,
    learning_rate_mapping={
        r"0\.sub_modules\.query\.0\.weight": 1e-3
    },  # Set a higher learning rate for the SparseStaticEmbedding module (see https://huggingface.co/blog/train-sparse-encoder#inference-free-splade)
    warmup_ratio=0.1,
    fp16=True,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=False,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    router_mapping={
        "query": "query",
        "answer": "document",
    },  # Map the column names to the routes
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=200,
    report_to="none",
)

# 6. Create a trainer & train
trainer = SparseEncoderTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

# 9. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

# 10. (Optional) Push it to the Hugging Face Hub
# model.push_to_hub(run_name)

# Evaluate

In [ ]:
from datasets import load_dataset
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sparse_encoder.evaluation import (
    SparseInformationRetrievalEvaluator,
)
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 1) Load Italian subsets
# params = {"path": "lopozz/CulturaViva-Retrieval", "split": "test"}
# names = ["corpus", "queries", "qrels"]
# params = {"path": "mteb/WikipediaRetrievalMultilingual", "split": "test"}
# names = ["it-corpus", "it-queries", "it-qrels"]
# params = {"path": "mteb/WebFAQRetrieval", "split": "test"}
# names = ["ita-corpus", "ita-queries", "ita-qrels"]
params = {"path": "mteb/MuPLeR-retrieval", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
for row in qrels_ds:
    qid = row["query-id"]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        relevant_docs.setdefault(qid, set()).add(cid)

# 4) Build evaluator
ir_evaluator = SparseInformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    show_progress_bar=True,
    batch_size=4,
    accuracy_at_k=[1],
    write_predictions=True
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
)

# 6) Run evaluation
results = ir_evaluator(model, output_path='../results/opensearch-project__opensearch-neural-sparse-encoding-multilingual-v1/predictions')
print(results)

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Corpus Chunks: 100%|██████████| 1/1 [02:56<00:00, 176.66s/it]

{'dot_accuracy@1': 0.425, 'dot_precision@1': 0.425, 'dot_precision@3': 0.18833333333333332, 'dot_precision@5': 0.122, 'dot_precision@10': 0.0665, 'dot_recall@1': 0.425, 'dot_recall@3': 0.565, 'dot_recall@5': 0.61, 'dot_recall@10': 0.665, 'dot_ndcg@10': 0.5425863798128064, 'dot_mrr@10': 0.5035873015873016, 'dot_map@100': 0.5127525880018967, 'query_active_dims': 30.75, 'query_sparsity_ratio': 0.9997095741365143, 'corpus_active_dims': 274.98040771484375, 'corpus_sparsity_ratio': 0.9974028805739114, 'avg_flops': 2.6418862342834473}


## MTEB Benchmark

### Load Tasks

In [1]:
import yaml
import mteb

with open("../configs/ds/mteg-retrieval-ita.yml", "r") as f:
    config = yaml.safe_load(f)

excluded = set(config["excluded_tasks"])

tasks = mteb.get_tasks(
    languages=config["languages"],
    modalities=config["modalities"],
    task_types=config["task_types"],
    exclusive_language_filter=True,
    eval_splits=["test"],
)

tasks = [t for t in tasks if t.metadata.name not in excluded]
tasks[:1]

/home/lpozzi/Git/lm-toolkit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[MuPLeRRetrieval(name='MuPLeR-retrieval', languages=['ita'])]

### Run Evaluation

In [ ]:
# use this to create preductinos

import mteb

cache = mteb.ResultCache("..")

model_name = "mteb/baseline-bm25s"
# model_name = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# model_name = "nickprock/splade-bert-base-italian-xxl-uncased-cv"

model_name = f"{model_name.split('/')[0]}__{model_name.split('/')[1]}"
model = mteb.get_model(model_name)
results = mteb.evaluate(
    model=model, 
    tasks=tasks[:1], 
    cache=cache, 
    encode_kwargs={"batch_size": 16},
    prediction_folder=f"../results/{model_name}/prediction_folder",
    overwrite_strategy="always"
)

print(results)

Evaluating task MuPLeR-retrieval:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating task MuPLeR-retrieval: 100%|██████████| 1/1 [00:11<00:00, 11.61s/it]

model_name='mteb/baseline-bm25s' model_revision='0_1_10' task_results=[TaskResult(task_name=MuPLeR-retrieval, main_score=0.79, scores=..., ...)] default_modalities=['text'] exceptions=[] experiment_name=None


### Performance Table

In [5]:
import mteb

model_names = [
    "mteb/baseline-bm25s",
    "nickprock/splade-bert-base-italian-xxl-uncased-cv",
    "google/embeddinggemma-300m",
    "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
]
cache = mteb.ResultCache("..")
results = cache.load_results(models=model_names, tasks=tasks, require_model_meta=False)

results.to_dataframe()

model_name,task_name,google/embeddinggemma-300m,mteb/baseline-bm25s,nickprock/splade-bert-base-italian-xxl-uncased-cv,opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1
0,MuPLeR-retrieval,0.84454,0.78597,0.510880,0.542586
1,WebFAQRetrieval,0.79523,0.55918,0.517111,0.554244
2,WikipediaRetrievalMultilingual,0.91761,0.84360,0.799550,0.837200


# Post Analysis

### Single Analysis

In [ ]:
from sentence_transformers import SparseEncoder, SparseEncoderModelCardData
from sentence_transformers.sentence_transformer.modules import Router, Transformer
from sentence_transformers.sparse_encoder.modules import (
    SparseStaticEmbedding,
    SpladePooling,
)

# 5) Load sparse model
# model_path = "../models/inference-free-splade-mmBERT-small-ita/final"
# model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
# model_path = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
model_path = "nickprock/splade-bert-base-italian-xxl-uncased-cv"
mlm_transformer = Transformer(model_path, transformer_task="fill-mask")

splade_pooling = SpladePooling(
    pooling_strategy="max",
    embedding_dimension=mlm_transformer.get_embedding_dimension(),
)
router = Router.for_query_document(
    query_modules=[
        SparseStaticEmbedding(tokenizer=mlm_transformer.tokenizer, frozen=False)
    ],
    # query_modules=[mlm_transformer, splade_pooling],
    document_modules=[mlm_transformer, splade_pooling],
)

model = SparseEncoder(
    modules=[router],
    model_card_data=SparseEncoderModelCardData(
        language="it",
        license="apache-2.0",
        model_name="Inference-free SPLADE mmBERT-small trained on Natural-Questions tuples",
    ),
    device='cpu'
)

In [ ]:
sentences = [
    "Per importare sostanze controllate nel corso del 2004, le imprese cui è stato assegnato un contingente devono richiedere alla Commissione, attraverso il sito Internet dell'ODS, una licenza di importazione utilizzando gli appositi moduli. Se i servizi della Commissione constatano che la richiesta è conforme alla quota autorizzata e in linea con i requisiti previsti dal regolamento (CE) n. 2037/2000, viene rilasciata una licenza di importazione. La Commissione si riserva il diritto di non concedere al licenza di importazione se la sostanza da importare non corrisponde alla descrizione o se rischia di non essere destinata all'uso autorizzato o ne è vietata l'importazione ai sensi del regolamento.",
    # "Che sole c'è fuori!",
    # "Ha guidato fino allo stadio.",
    # "The patient complained of severe cephalalgia"
]
embeddings = model.encode_document(sentences)
print(embeddings.shape)
# (3, 30522)

# Get the similarity scores for the embeddings
# similarities = model.similarity(embeddings, embeddings)
# print(similarities)
# tensor([[   32.4323,     5.8528,     0.0258],
#         [    5.8528,    26.6649,     0.0302],
#         [    0.0258,     0.0302,    24.0839]])

# Let's decode our embeddings to be able to interpret them
decoded = model.decode(embeddings, top_k=100)
for decoded, sentence in zip(decoded, sentences):
    print(f"Sentence: {sentence}")
    print(f"Decoded: {decoded}")
    print()

### Identify "Low Performers"
This function reads your predictions file and calculates NDCG@k for every query, returning the ones where the model performed a specific score (e.g. $NDCG@k = 0$).
Low performers analysis is carried out on a single dataset.

In [3]:
from datasets import load_dataset

task_name = 'MuPLeR-retrieval'
params = {"path": f"mteb/{task_name}", "split": "test"}
names = ["it-corpus", "it-queries", "it-qrels"]
corpus_ds = load_dataset(name=names[0], **params)
queries_ds = load_dataset(name=names[1], **params)
qrels_ds = load_dataset(name=names[2], **params)

# 2) Convert queries to {query_id: text} and corpus to {corpus_id: text}
id = 'id'
# id = '_id'
queries = {row[id]: row["text"] for row in queries_ds}
corpus = {row[id]: row["text"] for row in corpus_ds}

# 3) Convert qrels to {query_id: set(doc_ids)}
# If you want, you can keep only positives with score > 0
relevant_docs = {}
id2text = {}
queries_df = queries_ds.to_pandas()
for row in qrels_ds:
    qid = row["query-id"]
    qtext = queries_df[queries_df['id']==qid]['text'].iloc[0]
    cid = row["corpus-id"]
    score = row["score"]

    if score > 0:
        id2text[qid] = qtext
        relevant_docs.setdefault(qid, set()).add(cid)

In [4]:
import json
import math
import pandas as pd


def dcg_at_k(relevances, k):
    """
    DCG@k using the standard formulation:
    sum_i rel_i / log2(i + 2)

    i is zero-based, so rank 1 has denominator log2(2).
    """
    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevances[:k])
    )


def ndcg_at_k(retrieved_ids, gold_ids, k):
    """
    Binary nDCG@k:
    - relevance = 1 if retrieved doc is in gold_ids
    - relevance = 0 otherwise
    """
    gold_ids = set(gold_ids)

    if not gold_ids:
        return 0.0

    retrieved_relevances = [
        1 if cid in gold_ids else 0
        for cid in retrieved_ids[:k]
    ]

    dcg = dcg_at_k(retrieved_relevances, k)

    ideal_relevances = [1] * min(len(gold_ids), k)
    idcg = dcg_at_k(ideal_relevances, k)

    return dcg / idcg if idcg > 0 else 0.0


def get_performers(retrieval_predictions, relevant_docs, k=10, target_ndcg=0):
    performers = []

    
    for qid, results in retrieval_predictions['it']['test'].items():

        # Normalize prediction IDs to strings
        top_results = list(results.items())[:k]
        

        retrieved_ids = [str(id) for id, _ in top_results]
        retrieved_scores = [round(score, 2) for id, score in top_results]

        # Normalize gold IDs to strings
        gold_ids = sorted(str(cid) for cid in relevant_docs.get(qid, set()))

        ndcg = ndcg_at_k(
            retrieved_ids=retrieved_ids,
            gold_ids=gold_ids,
            k=k,
        )

        gold_scores = [
            round(results[cid], 2)
            for cid in gold_ids
            if results.get(cid) is not None
        ]

        performers.append({
            "qid": qid,
            # "query": data["query"],
            f"ndcg_at_{k}": ndcg,
            "retrieved_ids": retrieved_ids,
            "retrieved_scores": retrieved_scores,
            "gold_ids": gold_ids,
            "gold_scores": gold_scores,
        })

    df = pd.DataFrame(performers)

    return df[df[f"ndcg_at_{k}"] == target_ndcg].sort_values(by="qid")



model_name = "mteb/baseline-bm25s"
# model_name = "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1"
# model_name = "nickprock/splade-bert-base-italian-xxl-uncased-cv"

model_name = f"{model_name.split('/')[0]}__{model_name.split('/')[1]}"
with open(f'/home/lpozzi/Git/lm-toolkit/results/{model_name}/prediction_folder/{task_name}_predictions.json', 'r') as f:
    retrieval_predictions = json.load(f)

df_performers = get_performers(retrieval_predictions, relevant_docs, target_ndcg=1)
df_performers.insert(
    loc=df_performers.columns.get_loc("qid") + 1,
    column="qtext",
    value=df_performers["qid"].map(id2text)
)
df_performers

,qid,qtext,ndcg_at_10,retrieved_ids,retrieved_scores,gold_ids,gold_scores
67,05e24457-4e76-4ce9-8baf-53f988f5cc4c,Quale bulbo di regione delimitata è classifica...,1.0,"[7074, 9494, 8166, 8505, 8790, 9497, 9498, 400...","[17.06, 7.55, 7.51, 6.8, 6.78, 6.72, 6.61, 6.5...",[7074],[17.06]
158,07d4fd87-bfc7-4ae6-8bd3-ae5b1fd05607,Perché esortare gli Stati membri a stanziare i...,1.0,"[4387, 9515, 7505, 7992, 8746, 1482, 4398, 241...","[15.69, 9.77, 7.69, 6.95, 6.72, 6.67, 6.52, 6....",[4387],[15.69]
141,085de5cf-0266-48f8-9495-bfadae97ca80,Come un'impresa dominante preserva indirettame...,1.0,"[6973, 6961, 6960, 1892, 228, 2766, 8344, 6962...","[12.16, 9.92, 9.79, 8.65, 8.55, 8.38, 8.37, 8....",[6973],[12.16]
197,0a48517c-64f9-4965-8ac2-fbf813776d7c,Quale disposizione UE prevede copertura totale...,1.0,"[6639, 3200, 7438, 6592, 8675, 3593, 8623, 195...","[8.84, 6.63, 6.51, 6.45, 6.29, 6.23, 6.21, 6.1...",[6639],[8.84]
161,0b219bf5-ee9d-48ec-9529-6b8a050577ba,Perché sanzionare datori di lavoro discriminat...,1.0,"[385, 6324, 8047, 1757, 5145, 6848, 6851, 4889...","[12.3, 8.27, 8.08, 8.05, 7.51, 7.25, 6.77, 6.6...",[385],[12.3]
...,...,...,...,...,...,...,...
117,faa601d7-dea1-44d4-8bd9-3104d783cbaf,Quale frazione della catena di fornitura nazio...,1.0,"[9979, 7450, 9978, 3577, 9977, 614, 3511, 9930...","[16.19, 9.42, 6.25, 5.87, 5.68, 5.43, 4.92, 4....",[9979],[16.19]
61,fb5f0f61-88e6-4a8a-b176-ad142065c370,Quale materia prima aromatica è principalmente...,1.0,"[873, 886, 4610, 3242, 889, 5411, 4230, 8922, ...","[8.62, 6.84, 6.55, 5.52, 5.43, 5.24, 5.09, 5.0...",[873],[8.62]
101,fb7cd9ec-3c18-4af8-a814-ec045e33561a,Quale scenario di riferimento per l'UE a 25 pr...,1.0,"[2137, 2486, 4824, 2434, 4825, 1800, 2354, 704...","[21.28, 12.23, 7.89, 7.44, 7.37, 7.34, 7.24, 6...",[2137],[21.28]
122,fd24c534-2dca-468d-a854-2b4d58447739,Quando la compensazione è dimezzata salvo che ...,1.0,"[6197, 603, 5900, 566, 9607, 1002, 9072, 1810,...","[13.01, 6.97, 6.36, 6.01, 5.97, 5.97, 5.5, 5.3...",[6197],[13.01]


In [6]:
import json
import math
import pandas as pd
from pathlib import Path


def dcg_at_k(relevances, k):
    return sum(
        rel / math.log2(i + 2)
        for i, rel in enumerate(relevances[:k])
    )


def ndcg_at_k(retrieved_ids, gold_ids, k):
    gold_ids = set(str(x) for x in gold_ids)

    if not gold_ids:
        return 0.0

    retrieved_relevances = [
        1 if str(cid) in gold_ids else 0
        for cid in retrieved_ids[:k]
    ]

    dcg = dcg_at_k(retrieved_relevances, k)

    ideal_relevances = [1] * min(len(gold_ids), k)
    idcg = dcg_at_k(ideal_relevances, k)

    return dcg / idcg if idcg > 0 else 0.0


def normalize_model_name(model_name):
    org, name = model_name.split("/")
    return f"{org}__{name}"


def load_predictions(model_name, task_name, base_dir="/home/lpozzi/Git/lm-toolkit/results"):
    model_dir = normalize_model_name(model_name)

    path = Path(base_dir) / model_dir / "prediction_folder" / f"{task_name}_predictions.json"

    with open(path, "r") as f:
        return json.load(f)


def per_query_ndcg(
    retrieval_predictions,
    relevant_docs,
    model_label,
    k=10,
    lang="it",
    split="test",
):
    rows = []

    for qid, results in retrieval_predictions[lang][split].items():
        qid_str = str(qid)

        # Safer than relying on JSON order:
        # sort retrieved docs by score descending
        ranked_results = sorted(
            results.items(),
            key=lambda x: x[1],
            reverse=True,
        )[:k]

        retrieved_ids = [str(cid) for cid, _ in ranked_results]
        retrieved_scores = [round(score, 4) for _, score in ranked_results]

        gold_ids = sorted(str(cid) for cid in relevant_docs.get(qid, relevant_docs.get(qid_str, set())))

        score = ndcg_at_k(
            retrieved_ids=retrieved_ids,
            gold_ids=gold_ids,
            k=k,
        )

        rows.append({
            "qid": qid_str,
            f"{model_label}_ndcg@{k}": score,
            f"{model_label}_retrieved_ids": retrieved_ids,
            f"{model_label}_retrieved_scores": retrieved_scores,
            "gold_ids": gold_ids,
        })

    return pd.DataFrame(rows)


models = {
    "bm25": "mteb/baseline-bm25s",
    "opensearch_sparse": "opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1",
    "splade_it": "nickprock/splade-bert-base-italian-xxl-uncased-cv",
}

dfs = []

for label, model_name in models.items():
    preds = load_predictions(model_name, task_name)

    df_model = per_query_ndcg(
        retrieval_predictions=preds,
        relevant_docs=relevant_docs,
        model_label=label,
        k=10,
    )

    dfs.append(df_model)

df_compare = dfs[0]

for df in dfs[1:]:
    df_no_gold = df.drop(columns=["gold_ids"], errors="ignore")

    df_compare = df_compare.merge(
        df_no_gold,
        on="qid",
        how="inner",
    )

df_compare.insert(
    loc=df_compare.columns.get_loc("qid") + 1,
    column="qtext",
    value=df_compare["qid"].map(id2text),
)

df_compare.head()

,qid,qtext,bm25_ndcg@10,bm25_retrieved_ids,bm25_retrieved_scores,gold_ids,opensearch_sparse_ndcg@10,opensearch_sparse_retrieved_ids,opensearch_sparse_retrieved_scores,splade_it_ndcg@10,splade_it_retrieved_ids,splade_it_retrieved_scores
0,ba404e28-77f6-4dc0-8c45-b54c22053c83,Come si estendono le tutele fuori locali comme...,1.0,"[6099, 6954, 197, 8502, 8195, 2504, 5546, 2616...","[13.4167, 6.9343, 6.8391, 6.2509, 6.1587, 6.14...",[6099],0.289065,"[2616, 9938, 7519, 7697, 3214, 6954, 4235, 164...","[0.9381, 0.938, 0.9379, 0.9365, 0.9355, 0.9353...",0.000000,"[2825, 4888, 7463, 8394, 9964, 9970, 9827, 850...","[0.9659, 0.9633, 0.9624, 0.9621, 0.9619, 0.961..."
1,274beb3a-3ded-48f5-893c-b77181f358c9,Perché le imprese sottofinanziano la formazion...,1.0,"[8048, 5675, 8052, 9743, 6738, 2674, 3123, 258...","[10.8445, 7.0972, 6.9736, 6.579, 6.404, 6.2114...",[8048],0.000000,"[5285, 5487, 8054, 68, 1757, 80, 5972, 7561, 5...","[0.9329, 0.9313, 0.9295, 0.9284, 0.9282, 0.928...",0.289065,"[6834, 5538, 2586, 6306, 7460, 8157, 6455, 843...","[0.9607, 0.9589, 0.9583, 0.9572, 0.9563, 0.956..."
2,776d33f1-4283-40cb-9463-d4758a5a1ee7,Chi mantiene il potere di firma congiunta sul ...,1.0,"[4499, 9321, 2870, 7634, 9609, 6224, 5244, 375...","[17.7686, 7.3974, 5.5244, 5.5225, 5.4763, 5.47...",[4499],0.000000,"[7519, 5734, 7825, 5022, 6927, 312, 1647, 6478...","[0.9318, 0.928, 0.925, 0.9246, 0.9245, 0.9235,...",0.000000,"[7512, 4564, 5818, 3051, 2561, 428, 2240, 8702...","[0.9493, 0.9485, 0.948, 0.947, 0.9465, 0.9465,..."
3,eaf58e78-e2b7-43ec-b417-4106d5a6eea3,Quale organismo dovrebbe essere istituzionalme...,1.0,"[8583, 8587, 5627, 8595, 8586, 8591, 6632, 858...","[14.6599, 11.5913, 10.4247, 9.241, 8.8096, 8.7...",[8583],1.000000,"[8583, 5868, 1987, 8586, 3257, 2384, 2385, 668...","[0.9356, 0.9312, 0.9308, 0.9294, 0.9291, 0.928...",0.430677,"[9578, 3951, 7665, 8583, 5769, 1377, 7634, 504...","[0.9605, 0.9601, 0.9587, 0.9585, 0.9567, 0.956..."
4,5c849642-6405-4e6a-86ad-8a103649c485,Chi ha concluso che investitori buyout rafforz...,0.0,"[3370, 4886, 2155, 1707, 8349, 6501, 7745, 672...","[6.5714, 6.3976, 5.7079, 5.2298, 4.9371, 4.888...",[9831],0.000000,"[9938, 4234, 3762, 611, 5888, 4901, 3880, 2616...","[0.9487, 0.9462, 0.946, 0.946, 0.9453, 0.9445,...",0.000000,"[8331, 6720, 8937, 9946, 2872, 8347, 5818, 769...","[0.9639, 0.9589, 0.9568, 0.9565, 0.9564, 0.956..."


In [7]:
subset = df_compare[
    (df_compare["bm25_ndcg@10"] == 1.0)
    & (df_compare["opensearch_sparse_ndcg@10"] < 0.5)
    & (df_compare["splade_it_ndcg@10"] < 0.5)
].copy()

subset["bm25_margin_vs_best_other"] = (
    subset["bm25_ndcg@10"]
    - subset[["opensearch_sparse_ndcg@10", "splade_it_ndcg@10"]].max(axis=1)
)

subset = subset.sort_values(
    by="bm25_margin_vs_best_other",
    ascending=False,
)

subset

,qid,qtext,bm25_ndcg@10,bm25_retrieved_ids,bm25_retrieved_scores,gold_ids,opensearch_sparse_ndcg@10,opensearch_sparse_retrieved_ids,opensearch_sparse_retrieved_scores,splade_it_ndcg@10,splade_it_retrieved_ids,splade_it_retrieved_scores,bm25_margin_vs_best_other
5,cbd0e5c6-70f8-4f6d-b1cf-9b1dd5b7deeb,Quale organismo fornì una soluzione autonoma d...,1.0,"[3467, 3067, 5611, 9323, 4446, 3465, 8860, 965...","[8.8517, 7.0371, 6.1306, 5.9939, 5.9407, 5.675...",[3467],0.000000,"[5823, 8215, 2245, 6422, 636, 8611, 5711, 4859...","[0.9195, 0.9186, 0.9154, 0.9148, 0.9147, 0.913...",0.000000,"[5445, 1440, 7520, 4493, 2155, 4626, 5482, 910...","[0.9685, 0.9664, 0.9662, 0.966, 0.9656, 0.9655...",1.000000
2,776d33f1-4283-40cb-9463-d4758a5a1ee7,Chi mantiene il potere di firma congiunta sul ...,1.0,"[4499, 9321, 2870, 7634, 9609, 6224, 5244, 375...","[17.7686, 7.3974, 5.5244, 5.5225, 5.4763, 5.47...",[4499],0.000000,"[7519, 5734, 7825, 5022, 6927, 312, 1647, 6478...","[0.9318, 0.928, 0.925, 0.9246, 0.9245, 0.9235,...",0.000000,"[7512, 4564, 5818, 3051, 2561, 428, 2240, 8702...","[0.9493, 0.9485, 0.948, 0.947, 0.9465, 0.9465,...",1.000000
199,98a5a266-33f5-46a7-bef4-27a868d45e0f,Quale tipo di regolatore deve riesaminare peri...,1.0,"[2768, 3372, 4000, 3408, 1712, 636, 638, 643, ...","[16.2183, 7.6658, 7.571, 7.031, 6.7779, 6.7072...",[2768],0.000000,"[1434, 7826, 5211, 5036, 2766, 3924, 6998, 695...","[0.9542, 0.9526, 0.9498, 0.9479, 0.9476, 0.947...",0.000000,"[2229, 1167, 2790, 1448, 8118, 2780, 2766, 399...","[0.9701, 0.9697, 0.9697, 0.9697, 0.9685, 0.968...",1.000000
188,75a6f4cb-4279-4e3f-8af7-b2f4f475b0a7,Come la capacità addizionale dei concorrenti e...,1.0,"[6988, 3759, 5149, 5305, 2046, 5405, 6991, 832...","[18.5638, 7.546, 6.7933, 6.4079, 6.3317, 6.207...",[6988],0.000000,"[3248, 7016, 1627, 7618, 3164, 965, 6162, 5310...","[0.9342, 0.9306, 0.925, 0.919, 0.9189, 0.9174,...",0.000000,"[7748, 8331, 1194, 6022, 7752, 6177, 6913, 834...","[0.9666, 0.9657, 0.9647, 0.9637, 0.9634, 0.962...",1.000000
15,f82a0c60-b4a8-4261-a07b-300be0df8c2d,Quale modello analitico del 1983 è criticato p...,1.0,"[8609, 6771, 2684, 7366, 332, 3861, 1651, 5755...","[11.3683, 5.8178, 5.6141, 5.4365, 5.4051, 5.39...",[8609],0.000000,"[8708, 862, 8217, 5024, 3253, 9938, 5615, 2404...","[0.9422, 0.9396, 0.9396, 0.9385, 0.9363, 0.936...",0.000000,"[7450, 7697, 9946, 6977, 8331, 5373, 7692, 529...","[0.9671, 0.9657, 0.9646, 0.9643, 0.9629, 0.962...",1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,37f57533-dce1-4200-892c-f32a2b73a0b6,Quale comitato consultivo ha proposto una riun...,1.0,"[5723, 8203, 4012, 2454, 872, 5051, 8746, 3984...","[11.5837, 6.4006, 6.3072, 6.1964, 5.9687, 5.57...",[5723],0.430677,"[2647, 4074, 1063, 5723, 5569, 2468, 3400, 254...","[0.9292, 0.9264, 0.9247, 0.9223, 0.9172, 0.917...",0.000000,"[2708, 2474, 7968, 3887, 4486, 6185, 2466, 398...","[0.9668, 0.9666, 0.9663, 0.966, 0.9658, 0.9656...",0.569323
143,12dbc02e-bf1c-42b1-ac9c-a7485af40202,Quale comitato consultivo ha sollecitato un'ar...,1.0,"[1753, 1754, 4134, 1374, 2516, 4472, 1079, 987...","[12.7229, 12.1642, 9.9074, 9.6159, 9.4478, 8.8...",[1753],0.000000,"[3948, 2274, 6892, 5309, 6956, 6031, 9024, 917...","[0.9505, 0.9458, 0.9455, 0.945, 0.9447, 0.944,...",0.430677,"[7922, 5712, 6849, 1753, 5123, 1754, 3883, 968...","[0.9773, 0.9769, 0.9768, 0.9765, 0.9751, 0.975...",0.569323
150,11430392-723c-4e0f-a43c-a9f0d3845cbb,Quale autorizzazione istituzionale devono aver...,1.0,"[7534, 8994, 2753, 5014, 6059, 1606, 2755, 952...","[14.8843, 8.7524, 8.0603, 6.9403, 6.9296, 6.54...",[7534],0.430677,"[7506, 5650, 3018, 7534, 7794, 6578, 6046, 779...","[0.9297, 0.9278, 0.9211, 0.9206, 0.919, 0.9173...",0.000000,"[8012, 4143, 6834, 7247, 6850, 7246, 6048, 905...","[0.9565, 0.9538, 0.9536, 0.9535, 0.9534, 0.952...",0.569323
163,f2eacf7b-72d8-498b-9cd4-7c7c8ce3861f,Quale imposta sulle importazioni intra-UE era ...,1.0,"[3897, 8119, 4193, 811, 4837, 8693, 4867, 3531...","[7.9098, 7.

### Qualitative Inspection (FP vs FN)
Once you have a failure ID, this function prints a "Trial" view: what the model liked vs. what it should have liked.

In [ ]:
def qualitative_inspection(qid, df_fail, corpus):
    row = df_fail[df_fail['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}\n")
    print(f"❌ FALSE POSITIVES (Top 3 Model Choices):")
    for cid in row['retrieved_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")
        
    print(f"\n✅ FALSE NEGATIVES (What the model missed):")
    if not row['gold_ids']:
        print("  - No ground truth defined for this query.")
    for cid in row['gold_ids'][:3]:
        print(f"  - [ID {cid}]: {corpus[cid][:200]}...")

# Usage:
qualitative_inspection("05e24457-4e76-4ce9-8baf-53f988f5cc4c", df_performers, corpus)

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362]: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Dirett...
  - [ID 4494]: Per evitare ulteriori danneggiamenti che comporterebbero un calo di quantità di prodotto vendibile e soprattutto un calo di qualità dell'intera partita è determinante eseguire tale lavorazione in temp...
  - [ID 9014]: Nella pianificazione delle TEN-E si dovrebbe assegnare un mandato chiaro all'ENTSO-E e all'ACER nonché definire il ruolo di mediazione dell'UE. Il Libro verde non è sufficientemente esplicito su quest...

✅ FALSE NEGATIVES (What the model missed):
  - [ID 7074]: Le varietà Makói vöröshagyma o Makói hagyma sono classificate, imballate ed etichettat

### SPLADE Expansion Diagnosis
To see why a False Positive was ranked so high, we need to see the activated tokens. This function maps the sparse vector back to the vocabulary.

In [2]:
import torch

# 4. Map indices to words using the mlm_transformer tokenizer
VOCAB = {v: k for k, v in mlm_transformer.tokenizer.get_vocab().items()}

def diagnose_expansion(text, model, mode="query", top_n=-1):
    """
    Updated to use SparseEncoder's query and document specific encoding logic.
    """
    # 1. Run inference based on mode
    with torch.no_grad():
        if mode == "query":
            # SparseEncoder.encode_query returns a tensor (batch_size, vocab_size)
            vector = model.encode_query([text], convert_to_sparse_tensor=False)[0]
        else:
            # SparseEncoder.encode_document returns a tensor (batch_size, vocab_size)
            vector = model.encode_document([text], convert_to_sparse_tensor=False)[0]
    
    # 3. Move to CPU and convert to list
    weights = vector.cpu().tolist()
    
    activated = []
    for i, w in enumerate(weights):
        if w > 0: # Small epsilon to avoid float noise
            token = VOCAB.get(i, f"[UNK_{i}]")
            activated.append((token, w))
            
    # 5. Sort by weight descending
    sorted_activated = sorted(activated, key=lambda x: x[1], reverse=True)
    
    if top_n == -1:
        return sorted_activated
    return sorted_activated[:top_n]

def format_expansion(activations):
    """Formats the output of diagnose_expansion into a clean list."""
    for word, weight in activations:
        # Aligns word to the left (15 chars) and weight to 4 decimal places
        print(f"         {word:<15} | {weight:.4f}")

def show_intersection(query_weights, doc_weights, top_k=-1):
    """
    Fixed version: Converts list of tuples to dicts internally to handle 
    the intersection and weight lookups properly.
    """
    # Convert lists of (token, weight) tuples into dictionaries for lookup
    q_dict = dict(query_weights)
    d_dict = dict(doc_weights)
    
    # Find common tokens using set intersection on the dictionary keys
    common_tokens = set(q_dict.keys()) & set(d_dict.keys())
    
    # Build the intersection data
    intersection = {k: (q_dict[k], d_dict[k]) for k in common_tokens}
    
    # Sort by the product (contribution to total score)
    sorted_inter = sorted(intersection.items(), key=lambda x: x[1][0] * x[1][1], reverse=True)
    
    print(f"    Intersection ({len(sorted_inter)} tokens):")
    if not sorted_inter:
        print("      [No Overlap]")
    else:
        # Handle the top_k slicing (if -1, show everything)
        display_list = sorted_inter if top_k == -1 else sorted_inter[:top_k]
        
        for token, (qw, dw) in display_list:
            # We also calculate the product to show the actual impact on retrieval
            product = qw * dw
            print(f"      - {token:<15} | Q: {qw:.4f} * D: {dw:.4f} = Score: {product:.4f}")


def qualitative_expansion_inspection(qid, df_performers, corpus, queries, model):
    row = df_performers[df_performers['qid'] == qid].iloc[0]
    
    print(f"🔍 ANALYSIS FOR QUERY: {row['query']}")
    # query branch is not using SPLADE pooling
    #     token appears in query -> 1.0
    #     token not in query     -> 0.0
    query_expansion = diagnose_expansion(queries[qid], model, mode="query", top_n=-1)
    # format_expansion(query_expansion[:15]) # Show top 15 for query clarity

    # 1. Identify True Positives (Gold IDs that actually appear in Retrieved IDs)
    tps = [cid for cid in row['gold_ids'] if cid in row['retrieved_ids']]
    if tps:
        print(f"\n🚀 TRUE POSITIVES:")
        for cid in tps:
            rank = row['retrieved_ids'].index(cid) + 1
            score = row['retrieved_scores'][rank-1]
            print(f"  - [ID {cid}] | Rank: {rank} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)

    # 2. False Positives (Top of the list, but not in gold)
    print(f"\n❌ FALSE POSITIVES (Top 3 Model Choices):")
    for i, cid in enumerate(row['retrieved_ids'][:3]):
        if cid not in row['gold_ids']:
            score = row['retrieved_scores'][i]
            print(f"  - [ID {cid}] | Rank: {i+1} | Score: {score:.4f}")
            print(f"    Text: {corpus[cid][:500]}...")
            doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
            show_intersection(query_expansion, doc_expansion)
        
    # 3. False Negatives (Gold IDs that did NOT make it into the top k)
    cid  = row['gold_ids'][0]
    if cid not in row['retrieved_ids']:
        print(f"\n✅ FALSE NEGATIVES (What the model missed):")
        score = row['gold_scores'][0]
        print(f"  - [ID {cid}] | Score: {score:.4f} (Below Retrieval Threshold)")
        print(f"    Text: {corpus[cid][:500]}...")
        # Fixed: Define doc_expansion BEFORE showing intersection
        doc_expansion = diagnose_expansion(corpus[cid], model, mode="doc", top_n=-1)
        show_intersection(query_expansion, doc_expansion, top_k=10)
    else:
        print(f"\n✅ Zero FALSE NEGATIVES.")


import json
from pathlib import Path


def write_expansion_json(
    texts,
    model,
    model_name,
    output_path,
    mode="query",
    top_n=-1,
    ensure_ascii=False,
):
    output_path = Path(output_path)

    rows = []
    for idx, text in enumerate(texts):
        expansion = diagnose_expansion(
            text=text,
            model=model,
            mode=mode,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": float(weight),
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {
        model_name: rows
    }

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

In [43]:
for target_qid in df_performers['qid'].iloc[:1].tolist():
    qualitative_expansion_inspection(
        qid=target_qid,
        df_performers=df_performers,
        corpus=corpus,
        queries=queries,
        model=model,
    )

🔍 ANALYSIS FOR QUERY: Quale bulbo di regione delimitata è classificato sotto supervisione del produttore ma può essere reimballato fuori da quella regione?

❌ FALSE POSITIVES (Top 3 Model Choices):
  - [ID 7362] | Rank: 1 | Score: 4.7208
    Text: Benché l'allegato delinei una serie di procedure relative a contratti di appalto pre-commerciale che, pur non rientrando — in virtù di clausole di esclusione — nell'ambito di applicazione delle Direttive, sono però conformi al quadro giuridico vigente, esiste pur sempre la possibilità di una violazione, magari inconsapevole, di tale normativa. Il CESE raccomanda dunque ai committenti di esaminare attentamente l'allegato e di seguirne scrupolosamente le raccomandazioni. Nel caso in cui l'amminist...
    Intersection (5 tokens):
      - ##e             | Q: 1.0000 * D: 2.0762 = Score: 2.0762
      - deli            | Q: 1.0000 * D: 1.2698 = Score: 1.2698
      - ##o             | Q: 1.0000 * D: 0.6573 = Score: 0.6573
      - ##to            | Q

In [51]:
print(queries['ba404e28-77f6-4dc0-8c45-b54c22053c83'])
diagnose_expansion('Il medico ha prescritto un farmaco per ridurre la pressione alta.', model, mode="doc", top_n=-1)

Come si estendono le tutele fuori locali commerciali quando un investitore al dettaglio aderisce a fondo immobiliare chiuso per investire?


[('alta', 1.5419769287109375),
 ('farm', 1.3651257753372192),
 ('pressione', 1.3356409072875977),
 ('pres', 1.1909023523330688),
 ('rid', 1.1308432817459106),
 ('presion', 1.1208887100219727),
 ('##aco', 1.0940639972686768),
 ('alto', 1.0501487255096436),
 ('medico', 1.0243804454803467),
 ('haute', 0.9650628566741943),
 ('##ضغط', 0.9495495557785034),
 ('pression', 0.9311445951461792),
 ('cure', 0.9037303924560547),
 ('medicina', 0.9012002944946289),
 ('druck', 0.881564199924469),
 ('pressure', 0.8752270340919495),
 ('medica', 0.8582887649536133),
 ('فشار', 0.8335608243942261),
 ('tekanan', 0.815278172492981),
 ('cura', 0.7890410423278809),
 ('tratamiento', 0.763865053653717),
 ('tiba', 0.7505817413330078),
 ('درمان', 0.7021135091781616),
 ('treatment', 0.6995443105697632),
 ('pressao', 0.6894218325614929),
 ('tratar', 0.6880136132240295),
 ('for', 0.6726405024528503),
 ('reduce', 0.6724516153335571),
 ('hyper', 0.6702060699462891),
 ('##irat', 0.6388790011405945),
 ('##ac', 0.628410875

In [3]:
sentences = [
    "Il medico ha prescritto un farmaco per ridurre la pressione alta.",
    "Dopo l’incidente, l’auto è stata portata dal meccanico.",
    "Il ragazzo ha comprato un nuovo telefono perché il vecchio si era rotto.",
    "La banca ha approvato il prestito per l’acquisto della casa.",
    "Il cane correva felice nel parco durante una giornata di sole.",
    "La squadra ha perso la partita nonostante una prestazione eccellente.",
    "Non ho detto che Marco abbia rubato il portafoglio.",
    "Questo ristorante è una perla nascosta nel centro storico.",
    "Il professore ha spiegato la teoria della relatività con esempi semplici.",
    "La consegna del pacco è stata rimandata a causa dello sciopero dei trasporti.",
]

write_expansion_json(
    texts=sentences,
    model=model,
    model_name='splade-bert-base-italian-xxl-uncased-cv',
    output_path="splade_expansions.json",
    mode="doc",
    top_n=50,
)

In [ ]:
import json
from pathlib import Path

import bm25s


sentences = [
    "Il medico ha prescritto un farmaco per ridurre la pressione alta.",
    "Dopo l’incidente, l’auto è stata portata dal meccanico.",
    "Il ragazzo ha comprato un nuovo telefono perché il vecchio si era rotto.",
    "La banca ha approvato il prestito per l’acquisto della casa.",
    "Il cane correva felice nel parco durante una giornata di sole.",
    "La squadra ha perso la partita nonostante una prestazione eccellente.",
    "Non ho detto che Marco abbia rubato il portafoglio.",
    "Questo ristorante è una perla nascosta nel centro storico.",
    "Il professore ha spiegato la teoria della relatività con esempi semplici.",
    "La consegna del pacco è stata rimandata a causa dello sciopero dei trasporti.",
]


def build_bm25(sentences):
    corpus_tokens = bm25s.tokenize(sentences)
    retriever = bm25s.BM25()
    retriever.index(corpus_tokens)
    return retriever


def bm25_doc_expansion(text, doc_id, retriever, n_docs, top_n=-1):
    # Get readable token strings for this document
    tokens = bm25s.tokenize([text], return_ids=False)[0]

    expansion = []

    for token in sorted(set(tokens)):
        # Score this one token as a query against the whole corpus
        query_tokens = bm25s.tokenize(token)
        doc_ids, scores = retriever.retrieve(query_tokens, k=n_docs)

        # Find the score assigned to this document
        score_by_doc = dict(zip(doc_ids[0].tolist(), scores[0].tolist()))
        weight = float(score_by_doc.get(doc_id, 0.0))

        if weight > 0:
            expansion.append((token, weight))

    expansion = sorted(expansion, key=lambda x: x[1], reverse=True)

    if top_n != -1:
        expansion = expansion[:top_n]

    return expansion


def write_expansion_json_bm25(
    sentences,
    output_path,
    model_name="mteb/baseline-bm25s",
    mode="doc",
    top_n=-1,
    ensure_ascii=False,
):
    retriever = build_bm25(sentences)
    n_docs = len(sentences)

    rows = []

    for idx, text in enumerate(sentences):
        expansion = bm25_doc_expansion(
            text=text,
            doc_id=idx,
            retriever=retriever,
            n_docs=n_docs,
            top_n=top_n,
        )

        rows.append(
            {
                "id": idx,
                "text": text,
                "mode": mode,
                "expansion": [
                    {
                        "token": token,
                        "weight": weight,
                    }
                    for token, weight in expansion
                ],
            }
        )

    data = {model_name: rows}

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=ensure_ascii, indent=2)

    return data

In [7]:
data = write_expansion_json_bm25(
    sentences=sentences,
    output_path="bm25_expansions.json",
    top_n=-1,
)